# Database and FastAPI
Store three prices and retrieve one through a validated API.

## 1. Define a table

In [ ]:
from sqlalchemy import Float, String, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

class Base(DeclarativeBase):
    pass

class Price(Base):
    __tablename__ = "prices"
    id: Mapped[int] = mapped_column(primary_key=True)
    symbol: Mapped[str] = mapped_column(String)
    price: Mapped[float] = mapped_column(Float)

## 2. Create an in-memory database

In [ ]:
from sqlalchemy.pool import StaticPool

engine = create_engine(
    "sqlite://", connect_args={"check_same_thread": False}, poolclass=StaticPool
)
Base.metadata.create_all(engine)

## 3. Insert three rows

In [ ]:
with Session(engine) as session:
    session.add_all([
        Price(symbol="SPY", price=228.80),
        Price(symbol="QQQ", price=170.70),
        Price(symbol="GLD", price=140.11),
    ])
    session.commit()

## 4. Select one price

In [ ]:
with Session(engine) as session:
    spy = session.scalar(select(Price).where(Price.symbol == "SPY"))
    print(spy.symbol, spy.price)

## 5. Define the response

In [ ]:
from pydantic import BaseModel

class PriceResponse(BaseModel):
    symbol: str
    price: float

## 6. Create a first route

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/price/{symbol}")
def get_price(symbol: str):
    with Session(engine) as session:
        row = session.scalar(select(Price).where(Price.symbol == symbol))
        return None if row is None else PriceResponse(symbol=row.symbol, price=row.price)

## 7. Call it without a server

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)
response = client.get("/price/SPY")
print(response.status_code, response.json())

## 8. Deliberate limitation: unknown symbol

In [ ]:
missing = client.get("/price/UNKNOWN")
print(missing.status_code, missing.json())

## 9. Add a clear 404 route

In [ ]:
from fastapi import HTTPException

@app.get("/v2/price/{symbol}", response_model=PriceResponse)
def get_price_v2(symbol: str):
    with Session(engine) as session:
        row = session.scalar(select(Price).where(Price.symbol == symbol))
        if row is None:
            raise HTTPException(status_code=404, detail="Symbol not found")
        return PriceResponse(symbol=row.symbol, price=row.price)

## 10. Call the corrected route

In [ ]:
found = client.get("/v2/price/QQQ")
missing = client.get("/v2/price/UNKNOWN")
print(found.status_code, found.json())
print(missing.status_code, missing.json())

## 11. Final result

In [ ]:
assert found.json() == {"symbol": "QQQ", "price": 170.7}
assert missing.status_code == 404
print("Database-backed API works")

## Takeaways
- SQLAlchemy maps Python objects to rows.
- Pydantic describes the API response.
- TestClient exercises FastAPI without an external server.